In [ ]:
from pydantic import ValidationError
import os
import time
from groq import Groq
from pydantic import BaseModel
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)
class Advice(BaseModel):
    answer: str
    topic: str
    difficulty: str
history = []
system_prompt = {
    "role":"system",
    "content": """you are a tutor for school going kids. your ans should be short and easy to understand.You are a tutor for school-going kids.
    Return your response ONLY as JSON. The JSON must have exactly these fields: answer: string, topic: string, difficulty: string
    Do not add any other fields. Example:
    {
        "answer": "Gravity is a force that pulls objects toward each other.",
        "topic": "Physics",
        "difficulty": "Easy"
    }"""
}
history.append(system_prompt)
while(True):
    user_input = input("Enter your query ...")
    if ["stop", "exit", "break", "quit"].__contains__(user_input):
        break
    elif user_input == "reset":
        history = []
        history.append(system_prompt)
    user_input != "reset" and history.append({
        "role":"user",
        "content": "" if user_input == "reset" else user_input
    })
    try:
        if user_input != "reset":
            chat_completion = client.chat.completions.create(
                messages=history,
                model="llama-3.3-70b-versatile",
                stream=True
            )
            ai_ans=""
            for chunk in chat_completion:
                is_present = chunk.choices[0].delta.content == None
                if is_present == False:
                    print(chunk.choices[0].delta.content, end="", flush=True)
                    ai_ans += chunk.choices[0].delta.content
                    time.sleep(0.15)
            try:
                advice = Advice.model_validate_json(ai_ans)
                print(f"Data is fine{advice}")
            except ValidationError as e:
                print("AI response is not following expected model structure.")
            history.append({
                "role":"assistant",
                "content" : ai_ans
            })
    except Exception as e:
        print(f"Error{e}")

{
        "topic": "Physics",
        "difficulty": "Easy"
}AI response is not following expected model structure.
